<a href="https://colab.research.google.com/github/JaredLThompson/LLM_Misc/blob/main/notebooks/hello_droid_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Train the “Hello Droid” wake word


This is a restart-safe Colab adaptation of openWakeWord's automatic model-training notebook. It trains one model for **hello droid** and validates required files before expensive generation or training begins.

Run the cells in order. If any cell fails, stop and fix that failure before continuing; Colab otherwise permits later cells to produce misleading secondary errors.


# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup — safe to rerun

from pathlib import Path
import os
import subprocess
import sys

CONTENT = Path("/content")
OPENWAKEWORD = CONTENT / "openwakeword"
PIPER_GENERATOR = CONTENT / "piper-sample-generator"


def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)


# Clone the training repositories, or repair/update existing checkouts.
if (OPENWAKEWORD / ".git").is_dir():
    run(["git", "-C", OPENWAKEWORD, "fetch", "origin", "main"])
    run(["git", "-C", OPENWAKEWORD, "reset", "--hard", "origin/main"])
elif OPENWAKEWORD.exists():
    raise RuntimeError(
        f"{OPENWAKEWORD} exists but is not a Git checkout. "
        "Delete it in Colab's Files panel and rerun this cell."
    )
else:
    run(["git", "clone", "https://github.com/dscripka/openWakeWord.git", OPENWAKEWORD])

if not (PIPER_GENERATOR / ".git").is_dir():
    if PIPER_GENERATOR.exists():
        raise RuntimeError(
            f"{PIPER_GENERATOR} exists but is incomplete. "
            "Delete it in Colab's Files panel and rerun this cell."
        )
    run(["git", "clone", "https://github.com/rhasspy/piper-sample-generator", PIPER_GENERATOR])

piper_model = PIPER_GENERATOR / "models" / "en_US-libritts_r-medium.pt"
piper_model.parent.mkdir(parents=True, exist_ok=True)
if not piper_model.is_file() or piper_model.stat().st_size == 0:
    run([
        "wget", "-O", piper_model,
        "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",
    ])

# Install openWakeWord and the dependencies used by the upstream notebook.
run([sys.executable, "-m", "pip", "install", "-e", OPENWAKEWORD])
run([sys.executable, "-m", "pip", "install", "-e", PIPER_GENERATOR])
run([
    sys.executable, "-m", "pip", "install",
    "mutagen==1.47.0", "torchinfo==1.8.0",
    "torchmetrics==1.2.0", "speechbrain==0.5.14", "audiomentations==0.33.0",
    "torch-audiomentations==0.11.0", "acoustics==0.2.6", "pronouncing==0.2.0",
    "datasets==3.6.0", "deep-phonemizer==0.0.19",
])

# Download the feature-extraction models required by the training code.
resource_dir = OPENWAKEWORD / "openwakeword" / "resources" / "models"
resource_dir.mkdir(parents=True, exist_ok=True)
for filename in (
    "embedding_model.onnx",
    "embedding_model.tflite",
    "melspectrogram.onnx",
    "melspectrogram.tflite",
):
    destination = resource_dir / filename
    if not destination.is_file() or destination.stat().st_size == 0:
        run([
            "wget",
            "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/" + filename,
            "-O", destination,
        ])

config_path = OPENWAKEWORD / "examples" / "custom_model.yml"
train_path = OPENWAKEWORD / "openwakeword" / "train.py"
assert config_path.is_file(), f"Missing {config_path}"
assert train_path.is_file(), f"Missing {train_path}"
print("Environment ready:", OPENWAKEWORD)


In [ ]:
# Compatibility adapter for openWakeWord's legacy generator import.
# Current piper-sample-generator provides its implementation as a package,
# while openWakeWord train.py imports generate_samples from the repository root.
from pathlib import Path
import sys

PIPER_GENERATOR = Path("/content/piper-sample-generator")
PIPER_MODEL = PIPER_GENERATOR / "models" / "en_US-libritts_r-medium.pt"
PIPER_CONFIG = Path(str(PIPER_MODEL) + ".json")
PIPER_SHIM = PIPER_GENERATOR / "generate_samples.py"

assert PIPER_MODEL.is_file(), f"Missing {PIPER_MODEL}"
assert PIPER_CONFIG.is_file(), f"Missing {PIPER_CONFIG}"

PIPER_SHIM.write_text(
    '''from pathlib import Path
from piper_sample_generator.__main__ import generate_samples as _generate_samples

MODEL = Path(__file__).parent / "models" / "en_US-libritts_r-medium.pt"


def generate_samples(*args, **kwargs):
    kwargs.setdefault("model", MODEL)
    return _generate_samples(*args, **kwargs)
''',
    encoding="utf-8",
)

sys.path.insert(0, str(PIPER_GENERATOR))
from generate_samples import generate_samples

print("Modern Piper sample generator ready:", generate_samples)
print("CUDA available:", __import__("torch").cuda.is_available())


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio

from pathlib import Path
import numpy as np
import scipy.io.wavfile
from tqdm.auto import tqdm

# AudioSet is now published as Parquet rather than the tar shards used by
# the original openWakeWord notebook. Stream a bounded subset so Colab does
# not need to download the complete multi-terabyte dataset.
audioset_output = Path("/content/audioset_16k")
audioset_output.mkdir(parents=True, exist_ok=True)
target_audioset_clips = 1000

existing = list(audioset_output.glob("*.wav"))
if len(existing) < target_audioset_clips:
    audioset = datasets.load_dataset(
        "agkphysics/AudioSet",
        "balanced",
        split="train",
        streaming=True,
    )
    audioset = audioset.cast_column(
        "audio", datasets.Audio(sampling_rate=16000)
    )
    for index, row in enumerate(
        tqdm(audioset, total=target_audioset_clips, desc="AudioSet")
    ):
        if index >= target_audioset_clips:
            break
        audio = row["audio"]
        samples = np.asarray(audio["array"], dtype=np.float32)
        samples = np.clip(samples, -1.0, 1.0)
        scipy.io.wavfile.write(
            audioset_output / f"audioset-{index:05d}.wav",
            16000,
            (samples * 32767).astype(np.int16),
        )

audioset_files = list(audioset_output.glob("*.wav"))
assert audioset_files, "AudioSet streaming produced no WAV files"
print(f"AudioSet background clips ready: {len(audioset_files)}")

# Free Music Archive dataset
fma_output = Path("/content/fma")
fma_output.mkdir(parents=True, exist_ok=True)
target_fma_clips = 120  # approximately one hour of 30-second clips

existing = list(fma_output.glob("*.wav"))
if len(existing) < target_fma_clips:
    fma = datasets.load_dataset(
        "rudraml/fma", name="small", split="train", streaming=True
    )
    fma = fma.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for index, row in enumerate(
        tqdm(fma, total=target_fma_clips, desc="FMA")
    ):
        if index >= target_fma_clips:
            break
        audio = row["audio"]
        samples = np.asarray(audio["array"], dtype=np.float32)
        samples = np.clip(samples, -1.0, 1.0)
        scipy.io.wavfile.write(
            fma_output / f"fma-{index:05d}.wav",
            16000,
            (samples * 32767).astype(np.int16),
        )

fma_files = list(fma_output.glob("*.wav"))
assert fma_files, "FMA streaming produced no WAV files"
print(f"FMA background clips ready: {len(fma_files)}")


In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

The configuration below intentionally trains a single phrase, **hello droid**. A single target is easier to evaluate and tune than combining `hello droid` and `hey droid` in the first model.


In [ ]:
# Load the default training configuration using an absolute Colab path.
from pathlib import Path
import yaml

config_path = Path("/content/openwakeword/examples/custom_model.yml")
assert config_path.is_file(), (
    f"Missing {config_path}. Rerun the Environment Setup cell and do not continue "
    "until it succeeds."
)

with config_path.open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

config


In [ ]:
# Configure the Hello Droid model.
config["target_phrase"] = ["hello droid"]
config["model_name"] = "hello_droid"
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ["/content/audioset_16k", "/content/fma"]
config["false_positive_validation_data_path"] = "/content/validation_set_features.npy"
config["feature_data_files"] = {
    "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

training_config = Path("/content/hello_droid.yaml")
with training_config.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(config, stream, sort_keys=False)

required = [
    Path("/content/openwakeword/openwakeword/train.py"),
    Path("/content/validation_set_features.npy"),
    Path("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"),
    Path("/content/audioset_16k"),
    Path("/content/fma"),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing required training data:\n- " + "\n- ".join(missing)
print("Training configuration ready:", training_config)


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips.
import subprocess
import sys
from pathlib import Path

train_script = Path("/content/openwakeword/openwakeword/train.py")
training_config = Path("/content/hello_droid.yaml")
assert train_script.is_file(), f"Missing {train_script}"
assert training_config.is_file(), f"Missing {training_config}"
subprocess.run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--generate_clips",
], cwd="/content", check=True)


In [ ]:
# Step 2: Augment the generated clips.
subprocess.run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--augment_clips",
], cwd="/content", check=True)


In [ ]:
# Step 3: Train and export the model.
subprocess.run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--train_model",
], cwd="/content", check=True)


## Download the trained ONNX model

The training script exports the model beneath `/content/my_custom_model`. The next cell verifies it before starting the browser download.


In [ ]:
from pathlib import Path
from google.colab import files

model_path = Path("/content/my_custom_model/hello_droid.onnx")
assert model_path.is_file(), f"Training did not produce {model_path}"
assert model_path.stat().st_size > 100_000, f"Model appears incomplete: {model_path}"
print(f"Downloading {model_path.name} ({model_path.stat().st_size:,} bytes)")
files.download(str(model_path))
